# Work Package 1 — Combined Single Tissue Channel × Exp-4 (Late Fusion, Concat) — Colab Run

Advisor's own note: **"Tek kanal exp-4 ve exp-5 denenecek"** — 5 kanal yerine tum
dokular tek kanalda verilsin (all five tissue channels collapsed into ONE channel,
instead of concatenating them separately).

This notebook trains exp-4's `concat` late-fusion architecture with the tissue arm
seeing a **single combined channel** = per-pixel `max(granulation, slough, necrosis,
tendon, bone)` — tissue-**type** identity is discarded, only "how much evidence of
ANY non-healthy tissue type is here" survives. Everything else is unchanged from
exp-4's own `concat` run: same encoder (MiT-B1, ImageNet init), same CORN head, same
single-phase protocol (`freeze_epochs=0`), same lr/epochs/augmentation, same
`datasets/wound-staging-masks-exp-3/` 8-channel `.npy` data and the same verified
PIID 3-fold split exp-3/4/5 already use.

Writes `results.csv` with 3 rows (one per fold,
`config_id = mit_b1__corn__wp1_exp4concat__tissue=combined_max`), skip-if
-already-done, safe to interrupt and resume — matches exp-4's own `train.py` resume
pattern.

**Reference numbers to compare against** (already recorded, not re-derived here):
- RGB baseline (task_3, 3-fold mean): accuracy 0.8168, macro-F1 0.8182, QWK 0.9138
- Exp-4 concat, all 5 tissue channels separately fused (3-fold mean): accuracy
  0.7983, macro-F1 0.8003, QWK 0.9109

If the single combined channel matches or beats the 5-channel number, per-type
tissue identity isn't adding much beyond "where is something abnormal." If it's
clearly worse, the per-type distinction (which tissue, not just how much) is
carrying real signal that a single collapsed channel throws away.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Same `wound-staging-masks-exp-3.zip` as exp-3/exp-4/exp-5 — nothing new to upload
if you already put it on Drive for those runs.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/experiment_extended/wp1_single_tissue_channel/exp4_concat/scripts")
print(os.getcwd())

## 4b. Smoke-test sanity check

This WP reuses exp-4's own already shape/gradient-verified `model.py` unchanged —
only the tissue arm's input is now a combined max-channel instead of 5 concatenated
channels, same [B,1,H,W] shape either way — so no separate `test_*.py` was written
for it (kept to the brief's "minimal modification" instruction). Instead, run
`train.py`'s built-in `--smoke` mode (1 fold, 2 epochs, 8/4/4 samples) once to catch
any shape/IO issue before committing to the full 3-fold run.

In [ ]:
!python train.py --smoke --workers 2

## 5. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/wp1_exp4concat_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Train — 3 folds, combined-max tissue channel

`train.py` already skips any fold already present in `results.csv`, so the retry
loop below is safe exactly like exp-4's own notebook: if Colab disconnects
mid-run, just re-run this cell — checkpoints/results already point at Drive from
step 5, so nothing already finished is repeated.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("combined-max single-channel exp-4 run finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 7. Check results

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())

print("\nReference (already recorded, not re-derived here):")
print("  RGB baseline (3-fold mean):         accuracy=0.8168 macro_f1=0.8182 qwk=0.9138")
print("  Exp-4 concat, 5 channels separate:  accuracy=0.7983 macro_f1=0.8003 qwk=0.9109")